### Ce notebook permet de caractériser les segments issu de notre clustering

In [1]:

 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
 
# 0. Chargement des données déjà clusterisées

CHEMIN_FICHIER = "C:\\Users\\Ce PC\\client-scope-rfm-project\\notebooks\\clients_rfm_clusters.csv"
 
rfm = pd.read_csv(CHEMIN_FICHIER)
 
print(f"Données chargées : {rfm.shape[0]} clients, {rfm['cluster'].nunique()} segments")
 


Données chargées : 5853 clients, 4 segments


### Indicateur synthetiques des segments

In [2]:

# 1. Indicateurs business par segment

synthese = rfm.groupby("cluster").agg(
    nb_clients=("CustomerID", "count"),
    recence_moy=("recence_jours", "mean"),
    frequence_moy=("frequence", "mean"),
    ca_total=("montant_total", "sum"),
    ca_moyen_client=("montant_total", "mean"),
).reset_index()
 
# Panier moyen = montant total / nombre de commandes
synthese["panier_moyen"] = (synthese["ca_total"] /
                             (synthese["frequence_moy"] * synthese["nb_clients"]))
 

In [3]:

# Poids en % du nombre de clients et du CA total
synthese["% clients"] = (synthese["nb_clients"] / synthese["nb_clients"].sum() * 100)
synthese["% CA"] = (synthese["ca_total"] / synthese["ca_total"].sum() * 100)
 
# Indice de concentration : % CA / % clients
# > 1 = le segment rapporte proportionnellement plus qu'il ne pèse en nombre

synthese["indice_concentration"] = (synthese["% CA"] / synthese["% clients"]).round(2)
 

# 2. Risque de churn
#    -> % de clients du segment dont la récence dépasse le 75e
#       percentile global (seuil "client qui s'éloigne")

seuil_risque = rfm["recence_jours"].quantile(0.75)
 
risque = rfm.assign(en_risque=rfm["recence_jours"] > seuil_risque) \
            .groupby("cluster")["en_risque"].mean() * 100
synthese["% en risque de churn"] = synthese["cluster"].map(risque).round(1)
 

### Nommage des segments et recommandation d'action

In [4]:

# 3. Nommage et recommandation d'action
#    Basé sur la position relative de chaque segment (au-dessus ou
#    en-dessous des médianes globales de récence/fréquence/montant)

med_r = synthese["recence_moy"].median()
med_f = synthese["frequence_moy"].median()
med_m = synthese["ca_moyen_client"].median()
 
def nommer_segment(row):
    recent = row["recence_moy"] <= med_r
    frequent = row["frequence_moy"] >= med_f
    gros_montant = row["ca_moyen_client"] >= med_m
 
    if recent and frequent and gros_montant:
        return "Client_fidele", "Fidéliser en priorité : programme VIP, avant-premières, remerciement personnalisé"
    elif not recent and not frequent:
        return "À risque / inactifs", "Campagne de réactivation urgente (offre de retour, email de relance)"
    elif recent and not frequent:
        return "Nouveaux prometteurs", "Encourager le 2e achat : onboarding, offre de bienvenue ciblée"
    elif not recent and gros_montant:
        return "Gros clients qui s'éloignent", "Contact prioritaire : ces clients avaient de la valeur, à ne pas perdre"
    else:
        return "Clients réguliers", "Maintenir l'engagement : newsletters, programme de fidélité standard"
 
synthese[["segment", "action_recommandee"]] = synthese.apply(
    lambda row: pd.Series(nommer_segment(row)), axis=1
)


### Tableau de synthèse final

In [ ]:
 
# 4. Tableau de synthèse final

tableau_final = synthese[[
    "segment", "nb_clients", "% clients", "% CA", "indice_concentration",
    "panier_moyen", "recence_moy", "% en risque de churn", "action_recommandee"
]].round(1).sort_values("% CA", ascending=False)
 
tableau_final.columns = [
    "Segment", "Nb clients", "% clients", "% du CA", "Indice de valeur",
    "Panier moyen (£)", "Récence moy. (j)", "% à risque de churn", "Action recommandée"
]
 
print("\n=== Synthèse des segments pour décideurs ===\n")
print(tableau_final.to_string(index=False))
 
tableau_final.to_csv("../output/synthese_segments_decideurs.csv", index=False)
 


=== Synthèse des segments pour décideurs ===

            Segment  Nb clients  % clients  % du CA  Indice de valeur  Panier moyen (£)  Récence moy. (j)  % à risque de churn                                                                Action recommandée
      Client_fidele         947       16.2     71.1               4.4             577.2              42.5                  1.2 Fidéliser en priorité : programme VIP, avant-premières, remerciement personnalisé
      Client_fidele        1875       32.0     20.5               0.6             339.7              98.3                  3.3 Fidéliser en priorité : programme VIP, avant-premières, remerciement personnalisé
À risque / inactifs        1634       27.9      4.9               0.2             300.1             490.8                 84.6              Campagne de réactivation urgente (offre de retour, email de relance)
À risque / inactifs        1397       23.9      3.4               0.1             241.5             104.1            

### Quelques graphiques d'illustration

In [ ]:


# 5. Graphique à bulles : lisible en 1 coup d'œil
#    x = récence (plus à gauche = plus récent = mieux)
#    y = fréquence (plus haut = plus fidèle)
#    taille des bulles = poids en % du CA

fig, ax = plt.subplots(figsize=(8, 6))
 
couleurs = plt.cm.viridis(np.linspace(0, 1, len(synthese)))
 
for i, row in synthese.iterrows():
    ax.scatter(
        row["recence_moy"], row["frequence_moy"],
        s=row["% CA"] * 40,  # taille proportionnelle au poids en CA
        alpha=0.7, color=couleurs[i], edgecolors="black", linewidth=1
    )
    ax.annotate(
        f"{row['segment']}\n({row['% CA']:.0f}% du CA)",
        (row["recence_moy"], row["frequence_moy"]),
        textcoords="offset points", xytext=(0, 12),
        ha="center", fontsize=9, fontweight="bold"
    )
 
ax.invert_xaxis()  # récence faible (= client actif) affichée à droite
ax.set_xlabel("Récence moyenne (jours depuis dernier achat) — inversé : actif → à droite")
ax.set_ylabel("Fréquence moyenne (nb commandes)")
ax.set_title("Cartographie des segments clients\n(taille des bulles = poids dans le CA total)")
plt.tight_layout()
plt.savefig("../figures/cartographie_segments.png", dpi=150)
plt.close()
 
print("\nFichiers générés :")
print("- synthese_segments_decideurs.csv (tableau à intégrer dans un rapport/slide)")
print("- cartographie_segments.png (visuel pour présentation)")
 


Fichiers générés :
- synthese_segments_decideurs.csv (tableau à intégrer dans un rapport/slide)
- cartographie_segments.png (visuel pour présentation)


### Fin du pipeline